# GROUP BY in SQL

In this notebook, you’ll practice one of the most powerful tools in SQL: `GROUP BY`.

Grouping allows you to summarize data, calculate totals, compare categories, and uncover patterns that are not visible at the row level.

These hands-on exercises will help you build confidence using `GROUP BY` to turn raw data into meaningful insights.

Let’s dive in.


## **SQL Environment Setup (do not edit)**

In [1]:
# @title

%%capture
!mkdir -p notebook_lib
!wget -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules

import sqlite3
import pandas as pd
from pathlib import Path


In [2]:
# @title

DB_FILE = 'class.db'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON;")

conn.executescript(r'''
-- ============================================================
-- SQLite DDL + seed data for: category, employee, customer,
-- product, purchase, purchase_item
-- ============================================================

PRAGMA foreign_keys = ON;

-- Drop in dependency order
DROP TABLE IF EXISTS purchase_item;
DROP TABLE IF EXISTS purchase;
DROP TABLE IF EXISTS product;
DROP TABLE IF EXISTS customer;
DROP TABLE IF EXISTS employee;
DROP TABLE IF EXISTS category;

-- ----------------------------
-- 1) category
-- ----------------------------
CREATE TABLE category (
  category_id         INTEGER PRIMARY KEY,
  name                TEXT NOT NULL,
  description         TEXT,
  parent_category_id  INTEGER,
  FOREIGN KEY (parent_category_id) REFERENCES category(category_id)
);

INSERT INTO category (category_id, name, description, parent_category_id) VALUES
(1, 'Food', NULL, NULL),
(2, 'Beverages', NULL, NULL),
(3, 'Alcohols', 'Alcohol beverages, only for adults', 2),
(4, 'Alcohol-free beverages', NULL, 2),
(5, 'Fruits and vegetables', NULL, 1),
(6, 'Meat and fish', NULL, 1),
(7, 'Other', NULL, NULL);

-- ----------------------------
-- 2) employee  (note: you wrote "emplyee"; using employee)
-- ----------------------------
CREATE TABLE employee (
  employee_id  INTEGER PRIMARY KEY,
  last_name    TEXT NOT NULL,
  first_name   TEXT NOT NULL,
  birth_date   TEXT,  -- 'YYYY-MM-DD'
  hire_date    TEXT,  -- 'YYYY-MM-DD'
  address      TEXT,
  city         TEXT,
  country      TEXT,
  reports_to   INTEGER,
  FOREIGN KEY (reports_to) REFERENCES employee(employee_id)
);

INSERT INTO employee (
  employee_id, last_name, first_name, birth_date, hire_date, address, city, country, reports_to
) VALUES
(1, 'Aylett', 'Sigismondo', NULL, NULL, '43158 Stoughton Alley', 'Houston', 'United States', 1),
(2, 'Scholar', 'Haslett', NULL, '2020-10-27', '39 Pine View Junction', 'Burbank', 'United States', NULL),
(3, 'Sackes', 'Winona', NULL, '2019-04-14', '674 Chive Point', 'Saint Petersburg', 'United States', 1),
(4, 'Swainson', 'Michaela', '1982-10-08', '2020-04-29', NULL, NULL, NULL, 1),
(5, 'Mixer', 'Patsy', '1977-02-22', '2020-11-28', '34136 Cardinal Way', 'Tallahassee', 'United States', 2),
(6, 'Kraft', 'Dulcinea', '1975-10-15', NULL, '54 Mayfield Place', 'Washington', 'United States', 1),
(7, 'Dixer', 'Mona', '1970-04-30', NULL, NULL, NULL, NULL, 2),
(8, 'Meeron', 'Shaun', '1976-03-24', '2021-01-28', '9682 Dawn Alley', 'Irving', 'United States', 2),
(9, 'Irvine', 'Natale', NULL, '2020-02-10', '6382 Scofield Place', 'New York City', 'United States', 1),
(10, 'Chanter', 'Domenico', '1990-04-09', '2020-11-08', NULL, NULL, NULL, 2);

-- ----------------------------
-- 3) customer
-- ----------------------------
CREATE TABLE customer (
  customer_id    INTEGER PRIMARY KEY,
  contact_name   TEXT NOT NULL,
  company_name   TEXT,
  contact_email  TEXT NOT NULL,
  address        TEXT,
  city           TEXT,
  country        TEXT
);

INSERT INTO customer (
  customer_id, contact_name, company_name, contact_email, address, city, country
) VALUES
(1, 'Denny Guye', NULL, 'dguye0@weather.com', '8 5th Drive', 'Cincinnati', 'United States'),
(2, 'Ezmeralda Lockhart', NULL, 'elockhart1@intel.com', '51 Clyde Gallagher Trail', 'Stockton', 'United States'),
(3, 'Alley Baseke', 'Oloo', 'abaseke2@naver.com', NULL, NULL, NULL),
(4, 'Stacia Jobke', 'JumpXS', 'sjobke3@umn.edu', '7635 5th Point', 'Dallas', 'United States'),
(5, 'Eddy McGowan', 'Topicshots', 'emcgowan4@mac.com', NULL, NULL, NULL),
(6, 'Raquel Emes', NULL, 'remes5@ocn.ne.jp', '40 Melby Avenue', 'Waco', 'United States'),
(7, 'Nehemiah Gracey', NULL, 'ngracey6@go.com', '16 Hoepker Crossing', 'Racine', 'United States'),
(8, 'Swen O''Shevlan', 'Fliptune', 'soshevlan7@weibo.com', NULL, NULL, NULL),
(9, 'Reginald Heselwood', NULL, 'rheselwood8@woothemes.com', '363 Little Fleur Street', 'Dallas', 'United States'),
(10, 'Anneliese Privost', NULL, 'aprivost9@java.com', '467 Springview Alley', 'Knoxville', 'United States');

-- ----------------------------
-- 4) product
-- ----------------------------
CREATE TABLE product (
  product_id         INTEGER PRIMARY KEY,
  product_name       TEXT NOT NULL,
  category_id        INTEGER NOT NULL,
  quantity_per_unit  INTEGER,
  unit_price         REAL NOT NULL,
  units_in_stock     INTEGER,
  discontinued       TEXT, -- store as 't'/'f' (or NULL) exactly like your data
  FOREIGN KEY (category_id) REFERENCES category(category_id)
);

INSERT INTO product (
  product_id, product_name, category_id, quantity_per_unit, unit_price, units_in_stock, discontinued
) VALUES
(1,  'Parsnip', 5, NULL, 3.07, NULL, 't'),
(2,  'Beef - Striploin', 6, NULL, 11.61, NULL, 'f'),
(3,  'Beef - Top Butt', 6, NULL, 13.06, NULL, 't'),
(4,  'Cranberries - Dry', 5, NULL, 8.44, NULL, 'f'),
(5,  'Wine - Domaine Boyar Royal', 3, NULL, 21.98, 58, 'f'),
(6,  'Beer - Heinekin', 3, NULL, 2.32, 45, 'f'),
(7,  'Pasta - Orecchiette', 1, NULL, 3.50, 71, 'f'),
(8,  'Sun - Dried Tomatoes', 5, NULL, 3.81, 21, NULL),
(9,  'Cheese - Ermite Bleu', 1, NULL, 8.28, NULL, 't'),
(10, 'Tortillas - Flour, 12', 1, 12, 4.59, 81, 'f'),
(11, 'Kale - Red', 5, NULL, 1.22, NULL, 't'),
(12, 'Island Oasis - Banana Daiquiri', 3, NULL, 6.40, 61, 'f'),
(13, 'Longan', 5, NULL, 4.69, NULL, 't'),
(14, 'Tarts Assorted', 1, 5, 9.32, 66, 't'),
(15, 'Pepper - Chipotle, Canned', 1, NULL, 2.25, 79, 'f'),
(16, 'Split Peas - Green, Dry', 5, NULL, 5.33, 83, 'f'),
(17, 'Soda Water - Club Soda, 355 Ml', 4, NULL, 1.25, 81, NULL),
(18, 'Salmon Atl.whole 8 - 10 Lb', 6, NULL, 10.59, 5, 'f'),
(19, 'Bread - Roll, Soft White Round', 1, NULL, 2.00, 30, 't'),
(20, 'Wine - White, Colubia Cresh', 3, NULL, 12.24, NULL, 't'),
(21, 'Plums - Red', 5, NULL, 2.87, NULL, 't'),
(22, 'Pastry - Raisin Muffin - Mini', 1, 8, 7.79, 27, 'f'),
(23, 'Chips Potato Salt Vinegar 43g', 1, NULL, 2.41, 24, 'f'),
(24, 'Wine - Manischewitz Concord', 3, NULL, 13.99, NULL, 't'),
(25, 'Bagels Poppyseed', 1, 4, 4.43, 88, 't'),
(26, 'Oil - Sesame', 1, NULL, 4.88, NULL, 'f'),
(27, 'Longos - Chicken Caeser Salad', 6, NULL, 6.52, 13, 't'),
(28, 'Flower - Commercial Spider', 7, NULL, 15.95, 9, NULL),
(29, 'Bread - Calabrese Baguette', 1, NULL, 1.32, 35, 'f'),
(30, 'Sprouts - Pea', 5, NULL, 3.58, 1, 't'),
(31, 'Beef Wellington', 6, NULL, 27.12, 95, 't'),
(32, 'Brandy - Orange, Mc Guiness', 3, NULL, 25.66, 50, 't'),
(33, 'Wine - Lou Black Shiraz', 3, NULL, 16.37, 44, 't'),
(34, 'Crackers Cheez It', 1, NULL, 3.44, 75, 'f'),
(35, 'Wine - Remy Pannier Rose', 3, NULL, 6.68, 89, NULL),
(36, 'Tart Shells - Savory, 2', 1, NULL, 1.09, 89, 'f'),
(37, 'Seedlings - Clamshell', 7, NULL, 5.81, NULL, 'f'),
(38, 'Wine - Cotes Du Rhone Parallele', 3, NULL, 12.67, 85, 't'),
(39, 'Lychee - Canned', 1, NULL, 4.82, 46, 't'),
(40, 'Pasta - Spaghetti, Dry', 1, NULL, 1.59, 69, 'f');

-- ----------------------------
-- 5) purchase
-- ----------------------------
CREATE TABLE purchase (
  purchase_id    INTEGER PRIMARY KEY,
  customer_id    INTEGER NOT NULL,
  employee_id    INTEGER NOT NULL,
  total_price    REAL NOT NULL,
  purchase_date  TEXT NOT NULL,  -- 'YYYY-MM-DD HH:MM:SS'
  shipped_date   TEXT,           -- 'YYYY-MM-DD HH:MM:SS'
  ship_address   TEXT,
  ship_city      TEXT,
  ship_country   TEXT,
  FOREIGN KEY (customer_id) REFERENCES customer(customer_id),
  FOREIGN KEY (employee_id) REFERENCES employee(employee_id)
);

INSERT INTO purchase (
  purchase_id, customer_id, employee_id, total_price, purchase_date, shipped_date, ship_address, ship_city, ship_country
) VALUES
(6,  2,  4,  6.14, '2020-10-13 18:39:13', '2020-10-14 07:41:17', '51 Clyde Gallagher Trail', 'Stockton', 'United States'),
(10, 1,  1,  7.62, '2020-03-06 20:05:55', '2020-03-08 09:07:31', '8 5th Drive', 'Cincinnati', 'United States'),
(3,  4,  7,  6.96, '2021-02-16 02:09:26', '2021-02-16 18:11:54', NULL, NULL, NULL),
(2,  1,  6, 26.12, '2020-04-02 12:23:48', '2020-04-04 06:24:11', '8 5th Drive', 'Cincinnati', 'United States'),
(5,  1, 10, 17.72, '2020-11-09 19:26:33', '2020-11-11 00:27:07', '8 5th Drive', 'Cincinnati', 'United States'),
(7,  2,  7, 65.94, '2020-06-23 08:32:41', '2020-06-23 17:34:26', NULL, NULL, NULL),
(1,  1, 10,  9.64, '2020-12-05 10:22:59', '2020-12-06 09:25:32', '8 5th Drive', 'Cincinnati', 'United States'),
(4,  2,  2, 10.32, '2020-11-17 09:43:29', '2020-11-18 16:46:21', '51 Clyde Gallagher Trail', 'Stockton', 'United States'),
(8,  5,  4, 21.32, '2020-09-23 00:02:50', '2020-09-24 04:03:28', '2236 Basil Alley', 'Cincinnati', 'United States'),
(9,  7,  7, 14.00, '2020-07-12 17:08:15', '2020-07-13 22:10:31', '020 Dawn Circle', 'Carson City', 'United States');

-- ----------------------------
-- 6) purchase_item
-- ----------------------------
CREATE TABLE purchase_item (
  purchase_id  INTEGER NOT NULL,
  product_id   INTEGER NOT NULL,
  unit_price   REAL NOT NULL,
  quantity     INTEGER NOT NULL,
  PRIMARY KEY (purchase_id, product_id, unit_price, quantity),
  FOREIGN KEY (purchase_id) REFERENCES purchase(purchase_id),
  FOREIGN KEY (product_id) REFERENCES product(product_id)
);

INSERT INTO purchase_item (purchase_id, product_id, unit_price, quantity) VALUES
(6, 1, 3.07, 2),
(10, 8, 3.81, 2),
(3, 6, 2.32, 3),
(10, 1, 3.07, 4),
(10, 12, 6.40, 1),
(10, 3, 13.06, 3),
(10, 36, 1.09, 4),
(2, 3, 13.06, 2),
(5, 25, 4.43, 4),
(7, 5, 21.98, 3),
(10, 7, 3.50, 3),
(6, 26, 4.88, 2),
(7, 15, 2.25, 1),
(1, 23, 2.41, 4),
(4, 34, 3.44, 3),
(5, 14, 9.32, 2),
(6, 40, 1.59, 4),
(6, 37, 5.81, 1),
(3, 19, 2.00, 3),
(6, 39, 4.82, 4),
(8, 16, 5.33, 4),
(10, 33, 16.37, 1),
(5, 38, 12.67, 2),
(10, 30, 3.58, 1),
(9, 7, 3.50, 4),
(10, 22, 7.79, 2),
(10, 15, 2.25, 1),
(3, 4, 8.44, 1),
(1, 6, 2.32, 4),
(6, 25, 4.43, 1),
(6, 29, 1.32, 3),
(10, 21, 2.87, 3),
(10, 13, 4.69, 2),
(9, 11, 1.22, 2),
(5, 10, 4.59, 1),
(3, 39, 4.82, 3),
(5, 1, 3.07, 2),
(1, 21, 2.87, 4),
(1, 19, 2.00, 2),
(5, 20, 12.24, 1),
(3, 17, 1.25, 3),
(5, 17, 1.25, 4),
(10, 25, 4.43, 1),
(3, 23, 2.41, 2),
(6, 8, 3.81, 3),
(4, 28, 15.95, 1),
(6, 2, 11.61, 1),
(6, 15, 2.25, 3),
(6, 4, 8.44, 1),
(1, 3, 13.06, 3);

-- ============================================================
-- 🔹 ADDITIONAL DATA FOR ADVANCED GROUP BY / DISTINCT / RANKING
-- ============================================================

-- ------------------------------------------------------------
-- 1) Extra November 2020 purchases (for date filtering)
-- ------------------------------------------------------------

INSERT INTO purchase (
  purchase_id, customer_id, employee_id, total_price,
  purchase_date, shipped_date, ship_address, ship_city, ship_country
) VALUES
-- Same customer multiple November purchases (creates grouping strength)
(11, 1, 2, 20.00, '2020-11-03 10:00:00', '2020-11-06 12:00:00', '8 5th Drive', 'Cincinnati', 'United States'),
(12, 1, 2, 15.00, '2020-11-15 14:00:00', '2020-11-16 09:00:00', '8 5th Drive', 'Cincinnati', 'United States'),

-- Different customer same city (DISTINCT trap for city)
(13, 2, 4, 20.00, '2020-11-20 16:00:00', '2020-11-22 10:00:00', '51 Clyde Gallagher Trail', 'Stockton', 'United States'),

-- Intentional city mismatch (customer 4 lives in Dallas)
(14, 4, 6, 18.00, '2020-11-18 11:00:00', '2020-11-20 08:00:00', '7635 5th Point', 'Houston', 'United States'),

-- Tie creation for ranking (customer 2 total_spent tie scenario)
(15, 2, 7, 15.00, '2020-11-25 09:00:00', '2020-11-28 10:00:00', '51 Clyde Gallagher Trail', 'Stockton', 'United States');

-- ------------------------------------------------------------
-- 2) Extra purchase_item rows
-- ------------------------------------------------------------

-- Duplicate product in same purchase (COUNT vs COUNT DISTINCT difference)
INSERT INTO purchase_item (purchase_id, product_id, unit_price, quantity) VALUES
(11, 7, 3.50, 2),
(11, 7, 3.50, 1),   -- same product again (different quantity → allowed by PK)

-- Multiple products different categories
(12, 5, 21.98, 1),
(12, 6, 2.32, 3),

-- Add items to create category volume imbalance
(13, 31, 27.12, 1),
(13, 18, 10.59, 2),

-- Add variety for distinct_products per category
(14, 8, 3.81, 2),
(14, 13, 4.69, 1),

-- Add more sales for ranking differences
(15, 25, 4.43, 3),
(15, 34, 3.44, 2);

-- ------------------------------------------------------------
-- 3) Create intentional ranking tie (Practice 29)
-- ------------------------------------------------------------

-- Customer 3 extra purchase to match total_spent of customer 2
INSERT INTO purchase (
  purchase_id, customer_id, employee_id, total_price,
  purchase_date, shipped_date, ship_address, ship_city, ship_country
) VALUES
(16, 3, 1, 20.00, '2020-11-26 10:00:00', '2020-11-27 12:00:00', NULL, 'Dallas', 'United States');

INSERT INTO purchase_item (purchase_id, product_id, unit_price, quantity) VALUES
(16, 3, 13.06, 1),
(16, 21, 2.87, 2);

-- ------------------------------------------------------------
-- 4) Extra shipping delay variation (Practice 28)
-- ------------------------------------------------------------

INSERT INTO purchase (
  purchase_id, customer_id, employee_id, total_price,
  purchase_date, shipped_date, ship_address, ship_city, ship_country
) VALUES
-- Long delay (5 days)
(17, 6, 2, 12.00, '2020-10-01 09:00:00', '2020-10-06 09:00:00', '40 Melby Avenue', 'Waco', 'United States'),

-- Short delay (same day)
(18, 6, 2, 8.00, '2020-10-10 09:00:00', '2020-10-10 15:00:00', '40 Melby Avenue', 'Waco', 'United States');

INSERT INTO purchase_item (purchase_id, product_id, unit_price, quantity) VALUES
(17, 10, 4.59, 2),
(18, 29, 1.32, 3);
''')
print(f"Database ready ✅ ({DB_FILE})")


Database ready ✅ (class.db)


## Get to Know the Tables

In this notebook, we’ll work with the **Store Sales** database.

The database contains 6 tables. Before writing queries, take a moment to understand how they relate to each other.

### 🧍 1. `customer`

Contains information about store customers.

* `customer_id` – unique identifier (mandatory)
* `contact_name` – full name (mandatory)
* `company_name` – company name (optional)
* `contact_email` – email address (mandatory)
* `address` – address (optional)
* `city` – city (optional)
* `country` – country (optional)

---

### 📦 2. `product`

List of products available in the store.

* `product_id` – unique identifier (NOT NULL)
* `product_name` – product name (mandatory)
* `category_id` – links to `category` table (NOT NULL)
* `quantity_per_unit` – units per package (optional)
* `unit_price` – product price
* `units_in_stock` – available units (can be NULL)
* `discontinued` – TRUE if discontinued, FALSE otherwise (optional)

---

### 🏷️ 3. `category`

Product classification information.

* `category_id` – unique identifier (mandatory)
* `name` – category name (mandatory)
* `description` – optional description
* `parent_category_id` – references parent category (NULL if none)

---

### 🧾 4. `purchase`

Information about each order.

* `purchase_id` – unique identifier (mandatory)
* `customer_id` – customer placing the order (mandatory)
* `employee_id` – employee handling the order (mandatory)
* `total_price` – total order value (mandatory)
* `purchase_date` – timestamp of purchase (nullable)
* `shipped_date` – shipment timestamp (nullable)
* `ship_address` – shipping address (nullable)
* `ship_city` – shipping city (nullable)
* `ship_country` – shipping country (nullable)

---

### 🛒 5. `purchase_item`

Connects purchases with products.

* `purchase_id` – purchase reference (mandatory)
* `product_id` – product reference (mandatory)
* `unit_price` – price per unit
* `quantity` – number of units purchased

---

### 👩‍💼 6. `employee`

Employee information.

* `employee_id` – unique identifier (mandatory)
* `last_name` – last name (mandatory)
* `first_name` – first name (mandatory)
* `birth_date` – birth date (optional)
* `address` – address (optional)
* `city` – city (optional)
* `country` – country (optional)
* `reports_to` – manager’s employee_id (NULL if none)

---

### 🎯 Why this database is great for GROUP BY

This dataset allows you to answer questions like:

* How many purchases did each customer make?
* What is the total revenue per category?
* Which employee handled the most orders?
* What are the average sales per country?

In other words: perfect material for aggregation and grouping.

Let’s start exploring.


In [38]:
# @title Northwind Minimal ER Diagram
%%html
<img id="er-img" style="width:50%; max-width:100%; height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/group_by_practice_set.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/group_by_practice_set_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

In [3]:
# @title Practice 1
base_practice_1 = make_df_validator_nospoilers(
    expected_hash='c1e1faa06998490b36987b3c4877dace8edb249e2502a6ff2e1be341149c5895',
    required_cols=['purchase_id', 'category_name'],
    expected_rows=37,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_1 = base_practice_1

make_sql_runner(
    conn,
    runner_id="practice_1",
    description_md='### Practice 1\nFor each purchase, display all product categories. Display the following columns:\n\nThe ID of the purchase (name the column purchase_id).\nThe name of the category (name the column category_name).\nShow each category only once for each purchase.\n',
    validator=val_practice_1,
    sol_sql='SELECT DISTINCT purchase_id, category.name AS category_name FROM purchase_item JOIN product ON purchase_item.product_id = product.product_id JOIN category ON product.category_id = category.category_id;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item', 'product', 'category']
)


# Group By One Column


In [4]:
# @title Practice 2
base_practice_2 = make_df_validator_nospoilers(
    expected_hash='76c8dbcaa6277535c7315a9fc84a44276d13a7a9af6ad368a33b44230c603b81',
    required_cols=['name', 'average_unit_price'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_2 = base_practice_2

make_sql_runner(
    conn,
    runner_id="practice_2",
    description_md='### Practice 2 - The average unit price for each category\nFor each product category, show its name and find the average unit price. Display two columns: name and average_unit_price.\n',
    validator=val_practice_2,
    sol_sql='SELECT category.name, AVG(unit_price) AS average_unit_price FROM product JOIN category ON product.category_id = category.category_id GROUP BY category.name;',
    select_only=True,
    dedupe=True,
    schema_tables=['product', 'category']
)


In [5]:
# @title Practice 3
base_practice_3 = make_df_validator_nospoilers(
    expected_hash='b3491a9520bb2e3377eb0ec993c22b905750c1317a6aed7083b58b52d68252d8',
    required_cols=['customer_id', 'contact_name', 'purchases_number'],
    expected_rows=7,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_3 = base_practice_3

make_sql_runner(
    conn,
    runner_id="practice_3",
    description_md="### Practice 3 - Number of purchases for each customer\nCount the number of purchases made by each customer. Display the customer_id, contact_name, and purchases_number. Ignore the customers that aren't present in the purchase table.\n",
    validator=val_practice_3,
    sol_sql='SELECT purchase.customer_id, contact_name, COUNT(*) AS purchases_number FROM purchase JOIN customer ON purchase.customer_id = customer.customer_id GROUP BY purchase.customer_id, contact_name;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [6]:
# @title Practice 4
base_practice_4 = make_df_validator_nospoilers(
    expected_hash='46c3714316c5db53c09f8d9c2ba75e1857c785f22d21414e52917750a518a9f1',
    required_cols=['purchase_id', 'total_products_quantity'],
    expected_rows=18,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_4 = base_practice_4

make_sql_runner(
    conn,
    runner_id="practice_4",
    description_md='### Practice 4 - Total products quantity for each purchase\nFor each purchase, count the total product quantity.\n\nFor example, if the purchase has two products: a product with ID of 5 with quantity of 2, and a product with ID of 10 with quantity of one, the result will be 3. Display two columns: purchase_id and total_products_quantity.\n',
    validator=val_practice_4,
    sol_sql='SELECT purchase_id, SUM(quantity) AS total_products_quantity FROM purchase_item GROUP BY purchase_id;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item']
)


In [7]:
# @title Practice 5
base_practice_5 = make_df_validator_nospoilers(
    expected_hash='fad17ce26c334bbb1df7ca2e84981b3e34dc0c40f846e63198845c3080ecadc5',
    required_cols=['name', 'units'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_5 = base_practice_5

make_sql_runner(
    conn,
    runner_id="practice_5",
    description_md='### Practice 5 - Total number of units in stock for each category\nFor each category, count the total number of units in stock. Display two columns: name (the name of the category) and units (the total number of units in stock for this category)\n',
    validator=val_practice_5,
    sol_sql='SELECT name, SUM(units_in_stock) AS units FROM product JOIN category ON product.category_id = category.category_id GROUP BY name;',
    select_only=True,
    dedupe=True,
    schema_tables=['product', 'category']
)


In [8]:
# @title Practice 6
base_practice_6 = make_df_validator_nospoilers(
    expected_hash='8f084d99a88e19a4fcec93e4cb52bac590bb0342b5106f9221a38e4874f6409c',
    required_cols=['last_name', 'first_name', 'purchases_number'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_6 = base_practice_6

make_sql_runner(
    conn,
    runner_id="practice_6",
    description_md='### Practice 6 - Number of purchases for each employee\nFor each employee, find the number of purchases assigned to them. Display three columns – last_name, first_name, and purchases_number.\n',
    validator=val_practice_6,
    sol_sql='SELECT last_name, first_name, COUNT(purchase_id) AS purchases_number FROM purchase JOIN employee ON purchase.employee_id = employee.employee_id GROUP BY last_name, first_name;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'employee']
)


# Group By Multiple Columns


In [9]:
# @title Practice 7
base_practice_7 = make_df_validator_nospoilers(
    expected_hash='e76114ffe7d2aa6b2b6fedfca7a75e57b09cfa217449fffc98a6690a375c91f3',
    required_cols=['customer_id', 'employee_id', 'total_purchases_price'],
    expected_rows=13,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_7 = base_practice_7

make_sql_runner(
    conn,
    runner_id="practice_7",
    description_md='### Practice 7 - Total purchases price for (customer, employee) group\nFor each customer and employee, find the total price of purchases that are made by this customer and to which a given employee is assigned. Display three columns: customer_id, employee_id, and the total price of purchases. Rename the third column to total_purchases_price.\n',
    validator=val_practice_7,
    sol_sql='SELECT customer_id, employee_id, SUM(total_price) AS total_purchases_price FROM purchase GROUP BY customer_id, employee_id;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


In [10]:
# @title Practice 8
base_practice_8 = make_df_validator_nospoilers(
    expected_hash='d9d9392c18fbd0c03eb29b333d311a2f3e06645f7fab2c17895bb8f6712a7139',
    required_cols=['name', 'discontinued', 'minimum_price'],
    expected_rows=13,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_8 = base_practice_8

make_sql_runner(
    conn,
    runner_id="practice_8",
    description_md="### Practice 8 - Minimum unit price for two-columns group\nFor each category, find the minimum unit price among all discontinued products, the minimum unit price among all continued products, and the minimum unit price among all products that aren't known to be continued or discontinued. Display the following columns: name (the category name), discontinued, minimum_price.\n",
    validator=val_practice_8,
    sol_sql='SELECT name, discontinued, MIN(unit_price) AS minimum_price FROM product JOIN category ON product.category_id = category.category_id GROUP BY name, discontinued;',
    select_only=True,
    dedupe=True,
    schema_tables=['product', 'category']
)


In [11]:
# @title Practice 9
base_practice_9 = make_df_validator_nospoilers(
    expected_hash='c3d33ec43a80c291f82c8571438e760f59515364f8e3156825128b4d08b41910',
    required_cols=['ship_city', 'last_name', 'first_name', 'total_purchases_price'],
    expected_rows=13,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_9 = base_practice_9

make_sql_runner(
    conn,
    runner_id="practice_9",
    description_md="### Practice 9 - Total purchases prices for each city and employee\nFor each ship_city and each employee taking care of the purchases, find the total price for purchases shipped to this city (that are taken care of by this employee). The columns' names should be: ship_city, last_name, first_name, and total_purchases_price.\n\nNote: In such exercises you usually need to add an additional column in the GROUP BY clause – the identifier! If there were two or more employees sharing both names, then their total price for purchases would be combined into one row – and we'd have false data.\n",
    validator=val_practice_9,
    sol_sql='SELECT ship_city, last_name, first_name, SUM(total_price) AS total_purchases_price FROM purchase JOIN employee ON purchase.employee_id = employee.employee_id GROUP BY ship_city, last_name, first_name, employee.employee_id;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'employee']
)


In [12]:
# @title Practice 10
base_practice_10 = make_df_validator_nospoilers(
    expected_hash='2d5e1217d6939a583b8d8a50fceaf930f10ba076fc102649ee0411e8e32dd3a1',
    required_cols=['product_name', 'quantity', 'purchases_number'],
    expected_rows=56,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_10 = base_practice_10

make_sql_runner(
    conn,
    runner_id="practice_10",
    description_md='### Practice 10 - Purchases with products of given quantity\nFor each product and quantity, find the number of purchases that contain this quantity of this product. Display three columns: product_name, quantity, and purchases_number.\n',
    validator=val_practice_10,
    sol_sql='SELECT product_name, quantity, COUNT(purchase_id) AS purchases_number FROM purchase_item JOIN product ON purchase_item.product_id = product.product_id GROUP BY product_name, quantity;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item', 'product']
)


In [13]:
# @title Practice 11
base_practice_11 = make_df_validator_nospoilers(
    expected_hash='8eb935b7c218a875421f5926df2cf7cfd40a746b2c5fb4ca8fd2aea2fc92f871',
    required_cols=['customer_id', 'employee_id', 'ship_city', 'lowest_price', 'maximum_price', 'purchases_number'],
    expected_rows=14,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_11 = base_practice_11

make_sql_runner(
    conn,
    runner_id="practice_11",
    description_md='### Practice 11 - Lowest and highest prices, and the number of purchases\nFor each customer that made purchases, and employee who took care of these purchases, display the following information:\n\ncustomer_id\nemployee_id\nship_city\nThe price of the cheapest purchase – lowest_price\nThe price of the most expensive purchase – maximum_price\nThe number of purchases – purchases_number\n',
    validator=val_practice_11,
    sol_sql='SELECT customer_id, employee_id, ship_city, MIN(total_price) AS lowest_price, MAX(total_price) AS maximum_price, COUNT(*) AS purchases_number FROM purchase GROUP BY customer_id, employee_id, ship_city;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


# Having


In [14]:
# @title Practice 12
base_practice_12 = make_df_validator_nospoilers(
    expected_hash='a625f2cdd00d6cf0c6a5fcf29b8e822fa143e5b8a79dc378c0c4361852e0940f',
    required_cols=['contact_name', 'ship_address', 'purchases_quantity'],
    expected_rows=3,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_12 = base_practice_12

make_sql_runner(
    conn,
    runner_id="practice_12",
    description_md="### Practice 12 - Customers with more than one purchase\nFor each customer who made more than one purchase, display the customer's contact_name, the ship_address, and the number of purchases (purchases_quantity).\n",
    validator=val_practice_12,
    sol_sql='SELECT contact_name, ship_address, COUNT(*) AS purchases_quantity FROM purchase JOIN customer ON purchase.customer_id = customer.customer_id GROUP BY contact_name, ship_address HAVING COUNT(*) > 1;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [15]:
# @title Practice 13
base_practice_13 = make_df_validator_nospoilers(
    expected_hash='bd03fa1ee4fe97e50149f558237bba96806819aae577a47a4b5eaabebccbee1b',
    required_cols=['name', 'products_number'],
    expected_rows=1,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_13 = base_practice_13

make_sql_runner(
    conn,
    runner_id="practice_13",
    description_md="### Practice 13 - The number of products in category\nFor each product category, count the number of products for which the quantity_per_unit isn't NULL in the database. Display only the categories that have at least two such products. The result table should have two columns: name (the name of the category) and products_number.\n",
    validator=val_practice_13,
    sol_sql='SELECT name, COUNT(quantity_per_unit) AS products_number FROM product JOIN category ON product.category_id = category.category_id GROUP BY name HAVING COUNT(quantity_per_unit) >= 2;',
    select_only=True,
    dedupe=True,
    schema_tables=['product', 'category']
)


In [16]:
# @title Practice 14
base_practice_14 = make_df_validator_nospoilers(
    expected_hash='455cde1fbc183147b7cdcc3129398fe91347b0667a779a2ce961021b4fa59a12',
    required_cols=['ship_city', 'purchases_number', 'customers_number'],
    expected_rows=4,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_14 = base_practice_14

make_sql_runner(
    conn,
    runner_id="practice_14",
    description_md="### Practice 14 - The number of purchases and customers for each city\nFor each ship city, display the number of purchases and the number of customers who made purchases which were shipped to this city. Show data only for the cities to which there were at least two purchases shipped by at least two customers (or one customer at least two times). The columns' names should be: ship_city, purchases_number, and customers_number.\n",
    validator=val_practice_14,
    sol_sql='SELECT ship_city, COUNT(purchase_id) AS purchases_number, COUNT(customer_id) AS customers_number FROM purchase GROUP BY ship_city HAVING COUNT(purchase_id) >= 2 AND COUNT(customer_id) >= 2;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


In [17]:
# @title Practice 15
base_practice_15 = make_df_validator_nospoilers(
    expected_hash='799aefd9e6a38256efdbf7c8abf63a1a399728e7e4a586d167fc28fff878271e',
    required_cols=['customer_id', 'employee_id', 'minimum_price'],
    expected_rows=5,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_15 = base_practice_15

make_sql_runner(
    conn,
    runner_id="practice_15",
    description_md="### Practice 15 - Minimum price for (customer, employee) group\nFor all purchases ordered by a given customer and assigned to a given employee, find the minimum purchase price. Show only such (customer, employee) groups for which there are at least two purchases. The columns' names should be: customer_id, employee_id, and minimum_price.\n",
    validator=val_practice_15,
    sol_sql='SELECT customer_id, employee_id, MIN(total_price) AS minimum_price FROM purchase GROUP BY customer_id, employee_id HAVING COUNT(total_price) >= 2',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


In [18]:
# @title Practice 16
base_practice_16 = make_df_validator_nospoilers(
    expected_hash='d92760d253bcce6ac059f8a04c22478782df9031ce7b0ccd62ab9d6a6711b316',
    required_cols=['product_name', 'total_quantity'],
    expected_rows=7,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_16 = base_practice_16

make_sql_runner(
    conn,
    runner_id="practice_16",
    description_md='### Practice 16 - Total quantities\nFor each product, count the total quantity of this product among all purchases. Show only the products for which the total quantity is greater than 5, and that were ordered in at least three purchases. Display two columns: product_name and total_quantity.\n',
    validator=val_practice_16,
    sol_sql='SELECT product_name, SUM(quantity) AS total_quantity FROM purchase_item JOIN product ON purchase_item.product_id = product.product_id GROUP BY product_name HAVING SUM(quantity) > 5 AND COUNT(purchase_id) >= 3;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item', 'product']
)


# General exercises


In [19]:
# @title Practice 17
base_practice_17 = make_df_validator_nospoilers(
    expected_hash='c62d252af5c4aff0e9b04f074430d3d37dc454f4cc5bb198a47cc67ad3acf73f',
    required_cols=['ship_city', 'employees_number', 'average_total_price'],
    expected_rows=4,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_17 = base_practice_17

make_sql_runner(
    conn,
    runner_id="practice_17",
    description_md="### Practice 17 - Count the statistics for purchases\nFor each ship_city, display the number of employees who took care of the purchases to this city, and the average total price of these purchases. Show three columns: ship_city, employees_number, and average_total_price. Don't show data for the cities to which there was only one purchase shipped.\n",
    validator=val_practice_17,
    sol_sql='SELECT ship_city, COUNT(employee_id) AS employees_number, AVG(total_price) AS average_total_price FROM purchase GROUP BY ship_city HAVING COUNT(purchase_id) <> 1;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


In [20]:
# @title Practice 18
base_practice_18 = make_df_validator_nospoilers(
    expected_hash='f82dfc4eb57047092f1311836201ba70db0f49a216127c0346cf9653d6c51ab2',
    required_cols=['last_name', 'first_name'],
    expected_rows=0,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_18 = base_practice_18

make_sql_runner(
    conn,
    runner_id="practice_18",
    description_md='### Practice 18 - Filter the employees\nDisplay the last names and first names of employees who are assigned only to one purchase.\n',
    validator=val_practice_18,
    sol_sql='SELECT last_name, first_name FROM purchase JOIN employee ON purchase.employee_id = employee.employee_id GROUP BY last_name, first_name HAVING COUNT(purchase_id) = 1;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'employee']
)


In [21]:
# @title Practice 19
base_practice_19 = make_df_validator_nospoilers(
    expected_hash='e057cbc9d936bb052407739ca21faf26a27ffc36c92465ae9bd9b5c998ce13f8',
    required_cols=['contact_email'],
    expected_rows=5,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_19 = base_practice_19

make_sql_runner(
    conn,
    runner_id="practice_19",
    description_md='### Practice 19 - Filter the customers\nShow the emails (contact_email) of the customers who made at least two purchases or whose total price for all orders was greater than 20.\n',
    validator=val_practice_19,
    sol_sql='SELECT contact_email FROM purchase JOIN customer ON purchase.customer_id = customer.customer_id GROUP BY contact_email HAVING COUNT(purchase_id) >= 2 OR SUM(total_price) > 20;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [22]:
# @title Practice 20
base_practice_20 = make_df_validator_nospoilers(
    expected_hash='6b8ce4c2d698949431d747ace69d7c2d6a91afdbd208569cea73143e43933bc8',
    required_cols=['purchase_id', 'products_number', 'total_quantity'],
    expected_rows=18,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_20 = base_practice_20

make_sql_runner(
    conn,
    runner_id="practice_20",
    description_md='### Practice 20 - Number of products and total quantity for purchase\nFor each purchase, show the number of products and the quantity of all products items. Display three columns: purchase_id, products_number, and total_quantity.\n',
    validator=val_practice_20,
    sol_sql='SELECT purchase_id, COUNT(product_id) AS products_number, SUM(quantity) AS total_quantity FROM purchase_item GROUP BY purchase_id;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item']
)


# Group By with Where, Having, and Order by


In [23]:
# @title Practice 21
base_practice_21 = make_df_validator_nospoilers(
    expected_hash='601a3f8c1e405a818bc692676738f5b0f0fd59dae16137210caaf3ad4addd9df',
    required_cols=['manager_id', 'number_of_employees'],
    expected_rows=2,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_21 = base_practice_21

make_sql_runner(
    conn,
    runner_id="practice_21",
    description_md="### Practice 21 - Number of employees reporting to an employee\nHow many employees report to a given employee? For each employee to which someone reports, count the number of employees that report to them. Display two columns: manager_id and number_of_employees. Sort the results by the number of employees in descending order. Make sure the manager_id column in the result table doesn't have NULLs.\n",
    validator=val_practice_21,
    sol_sql='SELECT reports_to AS manager_id, COUNT(*) AS number_of_employees FROM employee WHERE reports_to IS NOT NULL GROUP BY reports_to ORDER BY COUNT(*) DESC;',
    select_only=True,
    dedupe=True,
    schema_tables=['employee']
)


In [24]:
# @title Practice 22
base_practice_22 = make_df_validator_nospoilers(
    expected_hash='4e3eccd488b6213d5df6dc92e3e2bfc3b2acb60b950bec863d935a5721eeec91',
    required_cols=['city', 'customers_quantity'],
    expected_rows=4,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_22 = base_practice_22

make_sql_runner(
    conn,
    runner_id="practice_22",
    description_md='### Practice 22 - The number of customers in cities\nFor each city except for Knoxville and Stockton, count how many customers live there. Sort the results by the city name in ascending order. Display two columns: city and customers_quantity.\n',
    validator=val_practice_22,
    sol_sql="SELECT city, COUNT(customer_id) AS customers_quantity FROM customer WHERE city <> 'Knoxville' AND city <> 'Stockton' GROUP BY city ORDER BY city;",
    select_only=True,
    dedupe=True,
    schema_tables=['customer']
)


In [25]:
# @title Practice 23
base_practice_23 = make_df_validator_nospoilers(
    expected_hash='a9cb5724f22561eb1f8a269d0bca95dae9c9af686bdfa15d58c1461c07924fba',
    required_cols=['contact_name', 'purchases_quantity'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_23 = base_practice_23

make_sql_runner(
    conn,
    runner_id="practice_23",
    description_md="### Practice 23 - The number of purchases\nFor each customer, count the number of purchases they made. Consider only the purchases with a non-NULL ship_city. Show only the customers whose total cost for all their purchases were more than 14. The columns' names should be contact_name and purchases_quantity. Sort the rows by contact_name.\n",
    validator=val_practice_23,
    sol_sql='SELECT contact_name, COUNT(*) AS purchases_quantity FROM purchase JOIN customer ON purchase.customer_id = customer.customer_id WHERE ship_city IS NOT NULL GROUP BY contact_name HAVING SUM(total_price) > 14 ORDER BY contact_name;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [26]:
# @title Practice 24
base_practice_24 = make_df_validator_nospoilers(
    expected_hash='9ce1877410d4f97dd8e61371165ea4fed59802e32d590d0a167cf42f4ab455e3',
    required_cols=['product_id', 'unit_price', 'purchases_number', 'total_quantity'],
    expected_rows=3,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_24 = base_practice_24

make_sql_runner(
    conn,
    runner_id="practice_24",
    description_md='### Practice 24 - Purchases and quantity for each product\nFor each product, display the product ID (product_id), the unit price of the product (unit_price), the number of times the product was purchased (as purchases_number) and the total quantity in which the product was purchased (as total_quantity). Display only the products with a unit price greater than 3.50 that were ordered more than twice. Sort the results by the total quantity in descending order and the number of purchases in descending order.\n',
    validator=val_practice_24,
    sol_sql='SELECT product_id, unit_price, COUNT(purchase_id) AS purchases_number, SUM(quantity) AS total_quantity FROM purchase_item WHERE unit_price > 3.50 GROUP BY product_id, unit_price HAVING COUNT(purchase_id) > 2 ORDER BY SUM(quantity) DESC, COUNT(purchase_id) DESC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item']
)


In [27]:
# @title Practice 25
base_practice_25 = make_df_validator_nospoilers(
    expected_hash='5c126779223ed0511fb7d8c80d248d90e1d01b24d994c931dca01e637be04fb5',
    required_cols=['purchase_id', 'total_items_rows', 'distinct_products'],
    expected_rows=18,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_25 = base_practice_25

make_sql_runner(
    conn,
    runner_id="practice_25",
    description_md='### Practice 25 - Total item rows vs distinct products per purchase\nFor each purchase, show:\n\n* purchase_id\n* total_items_rows: the number of rows in `purchase_item` for that purchase\n* distinct_products: the number of different `product_id` values in that purchase\n\nSort the results by:\n1) distinct_products (descending)\n2) total_items_rows (descending)\n3) purchase_id (ascending)\n',
    validator=val_practice_25,
    sol_sql='SELECT purchase_id, COUNT(*) AS total_items_rows, COUNT(DISTINCT product_id) AS distinct_products FROM purchase_item GROUP BY purchase_id ORDER BY distinct_products DESC, total_items_rows DESC, purchase_id ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item']
)


In [48]:
# @title Practice 26
base_practice_26 = make_df_validator_nospoilers(
    expected_hash='13f7c6878c1e21c60929be1b87c1b8a7dd6d91ff0d4763626ad0c4961c80efc6',
    required_cols=['ship_city', 'purchases_number', 'customers_number'],
    expected_rows=3,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_26 = base_practice_26

make_sql_runner(
    conn,
    runner_id="practice_26",
    description_md='### Practice 26 - Purchases vs distinct customers by ship city\nFor each `ship_city`, show:\n\n* ship_city\n* purchases_number: number of purchases shipped to that city\n* customers_number: number of different customers who had purchases shipped to that city\n\nRequirements:\n\n* Ignore purchases where `ship_city` is NULL.\n* Show only cities with at least 2 purchases.\n\nSort by:\n1) customers_number (descending)\n2) purchases_number (descending)\n3) ship_city (ascending)\n',
    validator=val_practice_26,
    sol_sql='SELECT ship_city, COUNT(*) AS purchases_number, COUNT(DISTINCT customer_id) AS customers_number FROM purchase WHERE ship_city IS NOT NULL GROUP BY ship_city HAVING COUNT(*) >= 2 ORDER BY customers_number DESC, purchases_number DESC, ship_city ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


In [47]:
# @title Practice 27
base_practice_27 = make_df_validator_nospoilers(
    expected_hash='d4d55a20467c9881233fa10c2a03f4cac68e6bd37ef2b6df2606e08d1b2cdf1a',
    required_cols=['contact_name', 'purchases_number'],
    expected_rows=4,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_27 = base_practice_27

make_sql_runner(
    conn,
    runner_id="practice_27",
    description_md='### Practice 27 - Purchases per customer (November 2020 only)\nConsider only purchases made in **November 2020**.\n\nFor each customer, show:\n\n* contact_name\n* purchases_number: the number of purchases they made in that period\n\nSort by:\n1) purchases_number (descending)\n2) contact_name (ascending)\n',
    validator=val_practice_27,
    sol_sql="SELECT c.contact_name, COUNT(*) AS purchases_number FROM purchase p JOIN customer c ON p.customer_id = c.customer_id WHERE p.purchase_date >= '2020-11-01' AND p.purchase_date < '2020-12-01' GROUP BY c.contact_name ORDER BY purchases_number DESC, c.contact_name ASC;",
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [46]:
# @title Practice 28
base_practice_28 = make_df_validator_nospoilers(
    expected_hash='9d9e50150b14db08d5c486e36031e870bbfe9b9ccf4ceae1495954e02cb89b22',
    required_cols=['employee_id', 'average_days_to_ship'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_28 = base_practice_28

make_sql_runner(
    conn,
    runner_id="practice_28",
    description_md='### Practice 28 - Average days to ship per employee\nConsider only purchases where `shipped_date` is NOT NULL.\n\nFor each employee, compute:\n\n* employee_id\n* average_days_to_ship: the average of (shipped_date - purchase_date) in **days**\n\nShow only employees who have at least **2** purchases with a non-NULL shipped_date.\n\nSort by average_days_to_ship (descending), then employee_id (ascending).\n',
    validator=val_practice_28,
    sol_sql='SELECT employee_id, AVG(julianday(shipped_date) - julianday(purchase_date)) AS average_days_to_ship FROM purchase WHERE shipped_date IS NOT NULL GROUP BY employee_id HAVING COUNT(*) >= 2 ORDER BY average_days_to_ship DESC, employee_id ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase']
)


In [45]:
# @title Practice 29
base_practice_29 = make_df_validator_nospoilers(
    expected_hash='5211d6d73f5691298533e93d1da003982ed0dc16a9bb7d72441cbec23789aaea',
    required_cols=['customer_id', 'contact_name', 'total_spent', 'purchases_number'],
    expected_rows=7,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_29 = base_practice_29

make_sql_runner(
    conn,
    runner_id="practice_29",
    description_md='### Practice 29 - Customer ranking by spending (tie-break rules)\nFor each customer who made purchases, compute:\n\n* customer_id\n* contact_name\n* total_spent: SUM(total_price)\n* purchases_number: COUNT(*)\n\nRank customers by:\n1) total_spent (descending)\n2) purchases_number (descending)\n3) customer_id (ascending)\n',
    validator=val_practice_29,
    sol_sql='SELECT c.customer_id, c.contact_name, SUM(p.total_price) AS total_spent, COUNT(*) AS purchases_number FROM purchase p JOIN customer c ON p.customer_id = c.customer_id GROUP BY c.customer_id, c.contact_name ORDER BY total_spent DESC, purchases_number DESC, c.customer_id ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [44]:
# @title Practice 30
base_practice_30 = make_df_validator_nospoilers(
    expected_hash='a9ca0d20c955ac7af82b518893cb9a6824efe7de8181661f72d8d3911b3962cd',
    required_cols=['purchase_id', 'customer_city', 'ship_city', 'purchase_date'],
    expected_rows=2,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_30 = base_practice_30

make_sql_runner(
    conn,
    runner_id="practice_30",
    description_md="### Practice 30 - City mismatch (customer vs shipping)\nFind purchases where the customer's `city` differs from the purchase `ship_city`.\n\nShow:\n\n* purchase_id\n* customer_city\n* ship_city\n* purchase_date\n\nRequirements:\n* Ignore rows where `ship_city` is NULL.\n* Ignore rows where the customer's city is NULL.\n\nSort by purchase_date (ascending), then purchase_id (ascending).\n",
    validator=val_practice_30,
    sol_sql='SELECT p.purchase_id, c.city AS customer_city, p.ship_city, p.purchase_date FROM purchase p JOIN customer c ON p.customer_id = c.customer_id WHERE p.ship_city IS NOT NULL AND c.city IS NOT NULL AND NOT (c.city = p.ship_city) ORDER BY p.purchase_date ASC, p.purchase_id ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase', 'customer']
)


In [43]:
# @title Practice 31
base_practice_31 = make_df_validator_nospoilers(
    expected_hash='349d1cec571980d7b751d9775a8931f0deae410506389c990d175afb6d89b0c5',
    required_cols=['category_name', 'employee_id'],
    expected_rows=70,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_31 = base_practice_31

make_sql_runner(
    conn,
    runner_id="practice_31",
    description_md='### Practice 31 - Checklist of every (category, employee) combination\nCreate a checklist that pairs **every category** with **every employee**.\n\nShow:\n\n * category_name\n* employee_id\n\nSort by category_name (ascending), then employee_id (ascending).\n',
    validator=val_practice_31,
    sol_sql='SELECT c.name AS category_name, e.employee_id FROM category c, employee e ORDER BY c.name ASC, e.employee_id ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['category', 'employee']
)


In [40]:
# @title Practice 32
base_practice_32 = make_df_validator_nospoilers(
    expected_hash='48ed313c5eb946ac15f9026c4827113d26c0cb982716e2ebaadd157982974b35',
    required_cols=['category_name', 'distinct_products', 'total_quantity_sold'],
    expected_rows=6,
    sort_rows=True,
    sort_cols=True,
    exact_cols=False,
    hide_missing_cols=True,
    hide_row_count=False,
)

val_practice_32 = base_practice_32

make_sql_runner(
    conn,
    runner_id="practice_32",
    description_md="""### Practice 32 - Distinct products and total quantity sold per category
For each category, show:

* category_name
* distinct_products: number of different products in that category that appear in purchase_item
* total_quantity_sold: total quantity sold across all purchase items for that category

Sort by total_quantity_sold (descending), then category_name (ascending).
""",
    validator=val_practice_32,
    sol_sql='SELECT c.name AS category_name, COUNT(DISTINCT p.product_id) AS distinct_products, SUM(pi.quantity) AS total_quantity_sold FROM purchase_item pi JOIN product p ON pi.product_id = p.product_id JOIN category c ON p.category_id = c.category_id GROUP BY c.name ORDER BY total_quantity_sold DESC, category_name ASC;',
    select_only=True,
    dedupe=True,
    schema_tables=['purchase_item', 'product', 'category']
)
